Restarted Python 3.12.1

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "raw_data"
CLEAN_DIR = BASE_DIR / "cleaned_data"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Current working folder:", BASE_DIR)
print("Raw data folder:", RAW_DIR)
print("Cleaned data folder:", CLEAN_DIR)

Current working folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard
Raw data folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard\raw_data
Cleaned data folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard\cleaned_data


In [ ]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Raw datasets loaded successfully.\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Raw datasets loaded successfully.

customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
order_items: 112,650 rows, 7 columns
payments: 103,886 rows, 5 columns
reviews: 99,224 rows, 7 columns
orders: 99,441 rows, 8 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
category_translation: 71 rows, 2 columns


In [ ]:
print("Orders preview:")
print(orders.head())

print("\nOrder items preview:")
print(order_items.head())

print("\nPayments preview:")
print(payments.head())

print("\nReviews preview:")
print(reviews.head())

print("\nProducts preview:")
print(products.head())

Orders preview:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00  

In [ ]:
print("Missing value check by dataset:")

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    missing_values = df.isna().sum().sort_values(ascending=False)
    print(missing_values[missing_values > 0].head(10))

# Notes from missing value check:
# - Review comment fields have many missing values, but they are not needed for this dashboard.
# - Some orders have missing delivery dates, likely because they were canceled, unavailable, or not delivered.
# - Some products have missing category fields, which will be filled as "unknown" later.

Missing value check by dataset:

CUSTOMERS
Series([], dtype: int64)

GEOLOCATION
Series([], dtype: int64)

ORDER_ITEMS
Series([], dtype: int64)

PAYMENTS
Series([], dtype: int64)

REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

SELLERS
Series([], dtype: int64)

CATEGORY_TRANSLATION
Series([], dtype: int64)


In [ ]:
orders[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].dtypes

order_purchase_timestamp         object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("Date columns converted successfully.\n")
print(orders[date_columns].dtypes)

Date columns converted successfully.

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [ ]:
payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg(order_payment_value=("payment_value", "sum"))
)

print("Original payments shape:", payments.shape)
print("Aggregated payments shape:", payments_agg.shape)

payments_agg.head()

Original payments shape: (103886, 5)
Aggregated payments shape: (99440, 2)


,order_id,order_payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [ ]:
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(review_score=("review_score", "mean"))
)

print("Original reviews shape:", reviews.shape)
print("Aggregated reviews shape:", reviews_agg.shape)

reviews_agg.head()

Original reviews shape: (99224, 7)
Aggregated reviews shape: (98673, 2)


,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [ ]:
# This step translates product categories from Portuguese to English.
# The products table contains product_category_name in Portuguese.
# The category_translation table maps each Portuguese category to an English category.
# A left join keeps all products, even if a translation is missing.
# The final product_category column is used in the dashboard for readable category analysis.

products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

products["product_category"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)

print("Product categories translated successfully.")

products[[
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_category"
]].head()

Product categories translated successfully.


,product_id,product_category_name,product_category_name_english,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,housewares


In [ ]:
# We start from order_items because this gives us one row per item sold.
# This is useful because one order can contain multiple products/sellers.

products_for_merge = products[[
    "product_id",
    "product_category"
]].drop_duplicates()

df = order_items.merge(orders, on="order_id", how="left")
print("After merging order_items + orders:", df.shape)

df = df.merge(customers, on="customer_id", how="left")
print("After merging customers:", df.shape)

df = df.merge(products_for_merge, on="product_id", how="left")
print("After merging products:", df.shape)

df = df.merge(sellers, on="seller_id", how="left")
print("After merging sellers:", df.shape)

df = df.merge(payments_agg, on="order_id", how="left")
print("After merging payments:", df.shape)

df = df.merge(reviews_agg, on="order_id", how="left")
print("After merging reviews:", df.shape)

df.head()

After merging order_items + orders: (112650, 14)
After merging customers: (112650, 18)
After merging products: (112650, 19)
After merging sellers: (112650, 22)
After merging payments: (112650, 23)
After merging reviews: (112650, 24)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category,seller_zip_code_prefix,seller_city,seller_state,order_payment_value,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,27277,volta redonda,SP,72.19,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,3471,sao paulo,SP,259.83,4.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,furniture_decor,37564,borda da mata,MG,216.87,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,perfumery,14403,franca,SP,25.78,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,garden_tools,87900,loanda,PR,218.04,5.0


In [ ]:
# Item revenue is the listed product price plus freight/shipping value.
df["item_revenue"] = df["price"] + df["freight_value"]

# Calculate total item revenue per order.
# This is needed because one order can have multiple items.
df["order_item_total_revenue"] = df.groupby("order_id")["item_revenue"].transform("sum")

# Allocate the order payment value across order items.
# This prevents double-counting revenue when an order has multiple products.
df["payment_value"] = np.where(
    df["order_item_total_revenue"] > 0,
    df["order_payment_value"] * (df["item_revenue"] / df["order_item_total_revenue"]),
    df["item_revenue"]
)

# Calculate delivery time in days.
df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

# Identify late deliveries.
# Only orders with actual delivery dates and estimated delivery dates are evaluated.
df["is_late_delivery"] = (
    df["order_delivered_customer_date"].notna()
    & df["order_estimated_delivery_date"].notna()
    & (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"])
).astype(int)

# Create date fields for dashboard filtering and trend charts.
df["order_year"] = df["order_purchase_timestamp"].dt.year
df["order_month"] = df["order_purchase_timestamp"].dt.month
df["order_month_name"] = df["order_purchase_timestamp"].dt.strftime("%b")
df["order_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

df[[
    "order_id",
    "order_item_id",
    "price",
    "freight_value",
    "item_revenue",
    "order_payment_value",
    "payment_value",
    "delivery_days",
    "is_late_delivery",
    "order_year_month"
]].head()

,order_id,order_item_id,price,freight_value,item_revenue,order_payment_value,payment_value,delivery_days,is_late_delivery,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,72.19,72.19,7.0,0,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,259.83,259.83,16.0,0,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,216.87,216.87,7.0,0,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,25.78,25.78,6.0,0,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,218.04,218.04,25.0,0,2017-02


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "raw_data"
CLEAN_DIR = BASE_DIR / "cleaned_data"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Current working folder:", BASE_DIR)
print("Raw data folder:", RAW_DIR)
print("Cleaned data folder:", CLEAN_DIR)

Current working folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard
Raw data folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard\raw_data
Cleaned data folder: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard\cleaned_data


In [ ]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Raw datasets loaded successfully.\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Raw datasets loaded successfully.

customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
order_items: 112,650 rows, 7 columns
payments: 103,886 rows, 5 columns
reviews: 99,224 rows, 7 columns
orders: 99,441 rows, 8 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
category_translation: 71 rows, 2 columns


In [ ]:
print("Orders preview:")
print(orders.head())

print("\nOrder items preview:")
print(order_items.head())

print("\nPayments preview:")
print(payments.head())

print("\nReviews preview:")
print(reviews.head())

print("\nProducts preview:")
print(products.head())

Orders preview:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00  

In [ ]:
print("Missing value check by dataset:")

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    missing_values = df.isna().sum().sort_values(ascending=False)
    print(missing_values[missing_values > 0].head(10))

# Notes from missing value check:
# - Review comment fields have many missing values, but they are not needed for this dashboard.
# - Some orders have missing delivery dates, likely because they were canceled, unavailable, or not delivered.
# - Some products have missing category fields, which will be filled as "unknown" later.

Missing value check by dataset:

CUSTOMERS
Series([], dtype: int64)

GEOLOCATION
Series([], dtype: int64)

ORDER_ITEMS
Series([], dtype: int64)

PAYMENTS
Series([], dtype: int64)

REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

SELLERS
Series([], dtype: int64)

CATEGORY_TRANSLATION
Series([], dtype: int64)
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description

In [ ]:
orders[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].dtypes

order_purchase_timestamp         object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("Date columns converted successfully.\n")
print(orders[date_columns].dtypes)

Date columns converted successfully.

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [ ]:
payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg(order_payment_value=("payment_value", "sum"))
)

print("Original payments shape:", payments.shape)
print("Aggregated payments shape:", payments_agg.shape)

payments_agg.head()

Original payments shape: (103886, 5)
Aggregated payments shape: (99440, 2)


,order_id,order_payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [ ]:
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(review_score=("review_score", "mean"))
)

print("Original reviews shape:", reviews.shape)
print("Aggregated reviews shape:", reviews_agg.shape)

reviews_agg.head()

Original reviews shape: (99224, 7)
Aggregated reviews shape: (98673, 2)


,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [ ]:
# This step translates product categories from Portuguese to English.
# The products table contains product_category_name in Portuguese.
# The category_translation table maps each Portuguese category to an English category.
# A left join keeps all products, even if a translation is missing.
# The final product_category column is used in the dashboard for readable category analysis.

products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

products["product_category"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)

print("Product categories translated successfully.")

products[[
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_category"
]].head()

Product categories translated successfully.


,product_id,product_category_name,product_category_name_english,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,housewares


In [ ]:
# We start from order_items because this gives us one row per item sold.
# This is useful because one order can contain multiple products/sellers.

products_for_merge = products[[
    "product_id",
    "product_category"
]].drop_duplicates()

df = order_items.merge(orders, on="order_id", how="left")
print("After merging order_items + orders:", df.shape)

df = df.merge(customers, on="customer_id", how="left")
print("After merging customers:", df.shape)

df = df.merge(products_for_merge, on="product_id", how="left")
print("After merging products:", df.shape)

df = df.merge(sellers, on="seller_id", how="left")
print("After merging sellers:", df.shape)

df = df.merge(payments_agg, on="order_id", how="left")
print("After merging payments:", df.shape)

df = df.merge(reviews_agg, on="order_id", how="left")
print("After merging reviews:", df.shape)

df.head()

After merging order_items + orders: (112650, 14)
After merging customers: (112650, 18)
After merging products: (112650, 19)
After merging sellers: (112650, 22)
After merging payments: (112650, 23)
After merging reviews: (112650, 24)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category,seller_zip_code_prefix,seller_city,seller_state,order_payment_value,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,27277,volta redonda,SP,72.19,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,3471,sao paulo,SP,259.83,4.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,furniture_decor,37564,borda da mata,MG,216.87,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,perfumery,14403,franca,SP,25.78,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,garden_tools,87900,loanda,PR,218.04,5.0


In [ ]:
# Item revenue is the listed product price plus freight/shipping value.
df["item_revenue"] = df["price"] + df["freight_value"]

# Calculate total item revenue per order.
# This is needed because one order can have multiple items.
df["order_item_total_revenue"] = df.groupby("order_id")["item_revenue"].transform("sum")

# Allocate the order payment value across order items.
# This prevents double-counting revenue when an order has multiple products.
df["payment_value"] = np.where(
    df["order_item_total_revenue"] > 0,
    df["order_payment_value"] * (df["item_revenue"] / df["order_item_total_revenue"]),
    df["item_revenue"]
)

# Calculate delivery time in days.
df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

# Identify late deliveries.
# Only orders with actual delivery dates and estimated delivery dates are evaluated.
df["is_late_delivery"] = (
    df["order_delivered_customer_date"].notna()
    & df["order_estimated_delivery_date"].notna()
    & (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"])
).astype(int)

# Create date fields for dashboard filtering and trend charts.
df["order_year"] = df["order_purchase_timestamp"].dt.year
df["order_month"] = df["order_purchase_timestamp"].dt.month
df["order_month_name"] = df["order_purchase_timestamp"].dt.strftime("%b")
df["order_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

df[[
    "order_id",
    "order_item_id",
    "price",
    "freight_value",
    "item_revenue",
    "order_payment_value",
    "payment_value",
    "delivery_days",
    "is_late_delivery",
    "order_year_month"
]].head()

,order_id,order_item_id,price,freight_value,item_revenue,order_payment_value,payment_value,delivery_days,is_late_delivery,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,72.19,72.19,7.0,0,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,259.83,259.83,16.0,0,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,216.87,216.87,7.0,0,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,25.78,25.78,6.0,0,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,218.04,218.04,25.0,0,2017-02


In [ ]:
dashboard = df[[
    "order_id",
    "order_item_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "seller_id",
    "seller_city",
    "seller_state",
    "product_id",
    "product_category",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "price",
    "freight_value",
    "item_revenue",
    "payment_value",
    "order_payment_value",
    "review_score",
    "is_late_delivery",
    "delivery_days",
    "order_month",
    "order_month_name",
    "order_year",
    "order_year_month"
]].copy()

dashboard = dashboard.rename(columns={
    "order_purchase_timestamp": "order_purchase_date"
})

# Handle missing values
dashboard["product_category"] = dashboard["product_category"].fillna("unknown")
dashboard["review_score"] = dashboard["review_score"].fillna(0)
dashboard["payment_value"] = dashboard["payment_value"].fillna(0)
dashboard["item_revenue"] = dashboard["item_revenue"].fillna(0)

print("Final dashboard table shape:", dashboard.shape)
print("Number of unique orders:", dashboard["order_id"].nunique())
print("Number of unique customers:", dashboard["customer_unique_id"].nunique())
print("Number of unique sellers:", dashboard["seller_id"].nunique())

dashboard.head()

Final dashboard table shape: (112650, 27)
Number of unique orders: 98666
Number of unique customers: 95420
Number of unique sellers: 3095


,order_id,order_item_id,customer_id,customer_unique_id,customer_city,customer_state,seller_id,seller_city,seller_state,product_id,...,item_revenue,payment_value,order_payment_value,review_score,is_late_delivery,delivery_days,order_month,order_month_name,order_year,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,48436dade18ac8b2bce089ec2a041202,volta redonda,SP,4244733e06e7ecb4970a6e2683c13e61,...,72.19,72.19,72.19,5.0,0,7.0,9,Sep,2017,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP,e5f2d52b802189ee658865ca93d83a8f,...,259.83,259.83,259.83,4.0,0,16.0,4,Apr,2017,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,5b51032eddd242adc84c38acab88f23d,borda da mata,MG,c777355d18b72b67abbeef9df44fd0fd,...,216.87,216.87,216.87,5.0,0,7.0,1,Jan,2018,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,9d7a1d34a5052409006425275ba1c2b4,franca,SP,7634da152a4610f1595efa32f14722fc,...,25.78,25.78,25.78,4.0,0,6.0,8,Aug,2018,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,df560393f3a51e74553ab94004ba5c87,loanda,PR,ac6c3623068f30de03045865e4e10089,...,218.04,218.04,218.04,5.0,0,25.0,2,Feb,2017,2017-02


In [ ]:
total_revenue = dashboard["payment_value"].sum()
total_orders = dashboard["order_id"].nunique()
average_order_value = total_revenue / total_orders
total_customers = dashboard["customer_unique_id"].nunique()
total_sellers = dashboard["seller_id"].nunique()

average_review_score = dashboard.loc[
    dashboard["review_score"] > 0,
    "review_score"
].mean()

# For late delivery rate, calculate at the order level instead of item level.
order_level_late_delivery = (
    dashboard
    .groupby("order_id", as_index=False)
    .agg(is_late_delivery=("is_late_delivery", "max"))
)

late_delivery_rate = order_level_late_delivery["is_late_delivery"].mean()

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: {average_order_value:,.2f}")
print(f"Total Customers: {total_customers:,}")
print(f"Total Sellers: {total_sellers:,}")
print(f"Average Review Score: {average_review_score:.2f}")
print(f"Late Delivery Rate: {late_delivery_rate:.2%}")

Total Revenue: 15,846,280.17
Total Orders: 98,666
Average Order Value: 160.61
Total Customers: 95,420
Total Sellers: 3,095
Average Review Score: 4.03
Late Delivery Rate: 7.93%


In [ ]:
print("Final validation checks\n")

print("Dashboard shape:", dashboard.shape)
print("Duplicate order item rows:", dashboard.duplicated(subset=["order_id", "order_item_id"]).sum())
print("Missing order IDs:", dashboard["order_id"].isna().sum())
print("Missing customer IDs:", dashboard["customer_id"].isna().sum())
print("Missing seller IDs:", dashboard["seller_id"].isna().sum())
print("Missing product IDs:", dashboard["product_id"].isna().sum())
print("Missing product categories:", dashboard["product_category"].isna().sum())
print("Missing payment values:", dashboard["payment_value"].isna().sum())
print("Negative payment values:", (dashboard["payment_value"] < 0).sum())
print("Negative price values:", (dashboard["price"] < 0).sum())
print("Negative freight values:", (dashboard["freight_value"] < 0).sum())

print("\nOrder status counts:")
print(dashboard["order_status"].value_counts())

print("\nDate range:")
print("First order date:", dashboard["order_purchase_date"].min())
print("Last order date:", dashboard["order_purchase_date"].max())

Final validation checks

Dashboard shape: (112650, 27)
Duplicate order item rows: 0
Missing order IDs: 0
Missing customer IDs: 0
Missing seller IDs: 0
Missing product IDs: 0
Missing product categories: 0
Missing payment values: 0
Negative payment values: 0
Negative price values: 0
Negative freight values: 0

Order status counts:
order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

Date range:
First order date: 2016-09-04 21:15:19
Last order date: 2018-09-03 09:06:57


In [ ]:
output_path = CLEAN_DIR / "dashboard_orders_summary.csv"

dashboard.to_csv(output_path, index=False)

print("Dashboard CSV created successfully.")
print("Output path:", output_path)
print(f"Rows exported: {len(dashboard):,}")
print(f"Columns exported: {len(dashboard.columns)}")

Dashboard CSV created successfully.
Output path: d:\Documents\Projects\amazon bie ecommerce\Project 1\project_1_bi_dashboard\cleaned_data\dashboard_orders_summary.csv
Rows exported: 112,650
Columns exported: 27
